# 03 Define Support Mask

Loads the FTH HDF5 result and defines the real-space support mask. Choose **one** method: Paint PNG, Napari, or circular support coordinates. Every method produces the same binary `supportmask`, which is saved as both PNG and in the HDF5 file.

In [ ]:
import os, sys
from os.path import join
from getpass import getuser

import h5py
import numpy as np
import matplotlib.pyplot as plt
import skimage.morphology
from pyFAI.detectors import Detector

def find_basefolder(start=None):
    folder = os.path.abspath(start or os.getcwd())
    for _ in range(8):
        if os.path.isdir(join(folder, "library")) and os.path.isdir(join(folder, "raw")):
            return folder
        parent = os.path.dirname(folder)
        if parent == folder:
            break
        folder = parent
    raise FileNotFoundError("Could not find beamtime root containing library/ and raw/.")

BASEFOLDER = find_basefolder()
sys.path.append(join(BASEFOLDER, "library"))
print("Beamtime root:", BASEFOLDER)
import fthcore as fth
import helper_functions as helper
import interactive
from interactive import cimshow
import mask_lib
import reconstruct_rb as rec
import PETRA_MaxP04_loading as loading
import fth_phase_workflow as wf
import painted_masks as pm

try:
    import cupy as cp
    import cupyx as cpx
    import CCI_core_cupy as cci
    import Phase_Retrieval as PhR
    GPU = True
    print("GPU available")
except Exception:
    import CCI_core as cci
    PhR = None
    GPU = False
    print("GPU unavailable")

%matplotlib widget
try:
    %load_ext jupyter_black
except Exception:
    pass

In [ ]:
BASEFOLDER = find_basefolder()
DATA_H5 = join(BASEFOLDER, "processed", "Logs", "data_recon_ImId_1269_rb.hdf5")
data = wf.load_data_dict(DATA_H5)
positive_label = data["positive_label"]
reference_label = data["reference_label"]
experimental_setup = data["experimental_setup"]
focus_fth = data.get("focus_fth", {})
prop_dist = focus_fth.get("prop_dist", 0)
phase = focus_fth.get("phase", 0)
recipe = data["mask_pixel_smooth_recipe"]
shape = data["holo"][positive_label]["image_c"].shape
mask_pixel_smooth = wf.butterworth_disk_mask(shape, recipe["radius"], recipe["order"])
print("Loaded:", DATA_H5)

## Build support-mask preview reconstruction

In [ ]:
pos = np.asarray(data["holo"][positive_label]["image_c"], dtype=float) / data["factor"]
ref = np.asarray(data["holo"][reference_label]["image_c"], dtype=float)
sum_c = pos + ref - data["offset"]
holo_for_support = sum_c * (1 - mask_pixel_smooth)
recon_support = wf.fth_reconstruct(
    holo_for_support,
    experimental_setup,
    fth,
    prop_dist=prop_dist,
    phase=phase,
)
recon_support = np.abs(recon_support)
cimshow(recon_support)

## Paint support mask with PNG

In [ ]:
# Option: export the reconstructed amplitude, paint support pixels bright red,
# save the edited image under the printed mask filename, then run the next cell.
support_png, support_painted_png = pm.mask_png_paths(
    BASEFOLDER,
    "supportmask",
    data["holo"][positive_label]["id"],
)
pm.save_mask_reference_png(
    recon_support,
    support_png,
    log_scale=True,
)
print("Paint bright red support pixels in:")
print(support_png)
print("Save the edited PNG as:")
print(support_painted_png)

In [ ]:
# Option: load bright-red pixels from the edited PNG as supportmask.
supportmask = pm.load_bright_red_mask_png(
    support_painted_png,
    expected_shape=recon_support.shape,
)
support_coordinates = []
sample = "paint_png"

fig, ax = plt.subplots(figsize=(6, 6))
vmin, vmax = np.percentile(recon_support, (1, 99))
ax.imshow(recon_support, vmin=vmin, vmax=vmax, cmap="gray")
ax.imshow(supportmask, alpha=0.4, cmap="binary")
ax.set_title("painted supportmask overlay")

## Napari support mask (alternative to Paint)

Run these two cells instead of the Paint or coordinate sections. Paint label 1 over the allowed real-space support; label 0 erases.

In [ ]:
try:
    import napari
    from napari.utils.colormaps import DirectLabelColormap
except ImportError as exc:
    raise ImportError("Install napari[all] to use this optional support-mask method.") from exc
%gui qt

initial_support = np.asarray(
    data.get("supportmask", np.zeros(recon_support.shape)), dtype=np.uint8
)
if initial_support.shape != recon_support.shape:
    initial_support = np.zeros(recon_support.shape, dtype=np.uint8)
try:
    support_viewer.close()
except NameError:
    pass
support_viewer = napari.Viewer(title="supportmask")
support_viewer.add_image(recon_support, name="FTH amplitude", colormap="gray")
support_layer = support_viewer.add_labels(
    initial_support, name="supportmask", opacity=0.5,
    colormap=DirectLabelColormap(color_dict={
        0: np.array([0.0, 0.0, 0.0, 0.0]),
        1: np.array([1.0, 0.0, 0.0, 1.0]),
        None: np.array([1.0, 0.0, 0.0, 1.0]),
    }),
)
support_layer.selected_label = 1
support_viewer.layers.selection.active = support_layer
print("Paint label 1 for allowed support and label 0 to erase.")
print("When finished, run the next cell.")

In [ ]:
supportmask = (np.asarray(support_layer.data) > 0).astype(np.uint8)
if supportmask.shape != recon_support.shape:
    raise ValueError(
        f"Napari support has shape {supportmask.shape}, expected {recon_support.shape}"
    )
support_coordinates = []
sample = "napari"
print(f"Napari support contains {int(supportmask.sum())} pixels.")

## Support coordinates / circle widget (alternative)

Run this section instead of Paint or Napari. Supply `(y, x, radius)` circles directly, or adjust them with the widget before creating the mask.

In [ ]:
def get_supportmask_coordinates(sample):
    """
    Dictionary that stores coordinates of circular support mask apertures.
    Taken from FTH_CDI_01.ipynb.
    """
    support_coord = dict()
    support_coord["s2408f"] = [
        (953.0, 929.0, 27.0),
        (955.5, 1045.5, 4.0),
        (887.5, 1023.5, 4.5),
        (1024.0, 1024.0, 4.5),
    ]
    support_coord["s2409a_d2"] = [
        (931.5, 898.0, 38.5),
        (932.0, 1054.0, 6.5),
        (839.5, 1024.0, 6.5),
        (1024.0, 1024.0, 6.5),
    ]
    support_coord["test"] = [
        (931.5, 897.5, 39.0),
        (931.5, 1054.0, 6.0),
        (839.5, 1024.0, 6.0),
        (1024.0, 1024.0, 6.0),
    ]
    support_coord["s2408f_b2_hr"] = [
        (396.0, 251.0, 108.5),
        (378.5, 737.5, 18.5),
        (87.5, 643.5, 17.0),
        (670.0, 650.0, 17.0),
    ]
    support_coord["s2409a_d2_hr"] = [
        (1077.5, 185.0, 153.0),
        (1029.5, 793.5, 12.5),
        (465.5, 330.5, 15.5),
        (670.0, 650.0, 14.0),
    ]
    support_coord["FGT"] = [
        (529.5, 481.0, 58.0),
        (525.3, 675.0, 5.0),
        (412.5, 636.0, 5.0),
        (640.0, 640.0, 5.0),
    ]
    support_coord["FGT_flipped"] = [
        (525.5, 483.5, 57.0),
        (526.5, 677.0, 5.0),
        (412.8, 641.0, 5.0),
        (640.0, 640.0, 5.0),
    ]
    support_coord["CuMnAs"] = [
        (650.0, 670.0, 7.0),
        (650.5, 43.0, 9.0),
        (291.0, 523.0, 9.0),
        (344.0, 533.0, 9.0),
        (387.5, 522.5, 9.0),
        (346.0, 356.0, 108.0),
    ]
    return support_coord[sample]


# Select stored coordinates, or replace this with your own list of
# (y, x, radius) tuples. Set False to use them directly without a widget.
sample = "CuMnAs"
support_coordinates = get_supportmask_coordinates(sample)
ADJUST_COORDINATES_WITH_WIDGET = True

if ADJUST_COORDINATES_WITH_WIDGET:
    print("Cover the object and reference apertures with circles.")
    ds_circle = interactive.InteractiveCircleCoordinates(
        recon_support,
        len(support_coordinates),
        coordinates=support_coordinates,
    )
else:
    print("Using support_coordinates directly without the widget.")

In [ ]:
if ADJUST_COORDINATES_WITH_WIDGET:
    support_coordinates = ds_circle.get_params()
supportmask = mask_lib.create_circle_supportmask(
    support_coordinates, recon_support.shape
)
supportmask = (supportmask > 0).astype(np.uint8)

fig, ax = plt.subplots(figsize=(6, 6))
vmin, vmax = np.percentile(recon_support, (1, 99))
ax.imshow(recon_support, vmin=vmin, vmax=vmax, cmap="gray")
ax.imshow(supportmask, alpha=0.4, cmap="binary")
ax.set_title("supportmask overlay")

## Define CDI ROI

In [ ]:
# Zoom/pan to the useful reconstruction area, then execute the next cell.
fig, ax = cimshow(supportmask.astype(int))

In [ ]:
roi_cdi_s = interactive.axis_to_roi(ax)
roi_cdi = wf.slices_to_roi(roi_cdi_s)
print("CDI ROI:", roi_cdi)

## Save support mask

In [ ]:
im_id = data["holo"][positive_label]["id"]
supportmask_png = pm.save_binary_mask_png(BASEFOLDER, "supportmask", im_id, supportmask)
for key in [
    "dark_id_im",
    "dark_id_topo",
    "im_id",
    "topo_id",
    "fth_hologram",
    "fth_hologram_unmasked",
    "fth_png_title",
    "fth_recon",
    "fth_recon_unmasked",
    "mask_pixel_smooth",
    "mask_multiplier",
    "sum_c",
    "diff_c",
    "mask_pixel_c",
    "mask_pixel_c_png",
    "prop_dist",
    "phase",
    "dx",
    "dy",
    "focus_operation",
    "roi",
    "recon_cdi",
    "recon_topo_cdi",
    "phase_retrieval_png",
    "roi_cdi",
    "retrieved_type",
    "phase_cdi",
    "prop_dist_cdi",
    "dx_cdi",
    "dy_cdi",
    "focus_mode_cdi",
    "roi_crop",
    "mask_bs_cdi",
]:
    data.pop(key, None)
data.update(
    {
        "supportmask": supportmask,
        "support_coordinates": np.asarray(support_coordinates, dtype=float),
        "support_sample": sample,
        "supportmask_png": supportmask_png,
    }
)
wf.save_data_dict(data, DATA_H5, overwrite=True)
print("Updated supportmask in:", DATA_H5)
print("Saved supportmask PNG:", supportmask_png)

In [ ]:
# Workflow summary
_summary_data = data if "data" in globals() and isinstance(data, dict) else {}
_summary_h5 = globals().get("DATA_H5", _summary_data.get("data_file", "n/a"))
_summary_holo = _summary_data.get("holo", {})
_summary_pos = _summary_data.get(
    "positive_label", globals().get("positive_label", None)
)
_summary_ref = _summary_data.get(
    "reference_label", globals().get("reference_label", None)
)
_summary_im = _summary_holo.get(_summary_pos, {}).get(
    "id", globals().get("im_id", "n/a")
)
_summary_topo = _summary_holo.get(_summary_ref, {}).get(
    "id", globals().get("topo_id", "n/a")
)
print("im_id:", _summary_im)
print("topo_id:", _summary_topo)
print("HDF5:", _summary_h5)